# S0E3 lab · Action tokens, by hand

**Video:** S0E3 · Your first VLA, end to end on [Foundation Models for Robotics](https://www.youtube.com/@FoundationModelsForRobotics)
**Book:** [Action Models for Robot Learning, Ch 2](https://action-models-book.vercel.app/chapters/2/1/)
**You need:** Python basics. No GPU. About 25 minutes.

OpenVLA doesn't output numbers directly. It outputs tokens, and 256 of its tokens each stand for one bin of movement. You'll build that tokenizer, push a real FR3 trajectory through it, and measure what the bins cost.

In [ ]:
# Setup: works in Google Colab and locally (after `pip install -e .` in the repo root)
import os, sys
if "google.colab" in sys.modules:
    os.environ["MUJOCO_GL"] = "egl"
    if not os.path.exists("fmr-robot-learning-labs"):
        !git clone -q https://github.com/kandapagari/fmr-robot-learning-labs.git
    %cd fmr-robot-learning-labs
    !pip install -q -e .
    sys.path.insert(0, "labs/s0e3-action-tokens")
else:
    sys.path.insert(0, os.path.abspath("."))
import numpy as np, matplotlib.pyplot as plt, mediapy as media
import fmr6, action_tokens as at

## 1 · A trajectory to tokenize

The FR3's hand moves along a smooth S-curve. A policy would output this as a sequence of small steps (deltas): how far to move in x, y and z on each step.

In [ ]:
robot = fmr6.Robot("table", arm="fr3")
path = at.record_trajectory(robot)                 # (121, 3) fingertip positions, metres
deltas = np.diff(path, axis=0)                     # (120, 3) one step each
print("steps:", len(deltas), " typical step size (mm):", (np.linalg.norm(deltas, axis=1).mean() * 1000).round(2))

## 2 · Statistics, then bins

OpenVLA keeps the 1st and 99th percentile of every action number from its training data, and spreads 256 bins evenly between them. Anything outside gets clipped to the end bins.

In [ ]:
stats = at.statistics(deltas)
print("q01 (mm):", (stats["q01"] * 1000).round(3))
print("q99 (mm):", (stats["q99"] * 1000).round(3))
print("bin width with 256 bins (mm):", at.bin_width_mm(stats).round(4))
tokens = at.tokenize(deltas, stats)
print("first 5 steps as tokens:", tokens[:5].tolist())

Each step became three integers between 0 and 255. In OpenVLA those integers map onto the 256 least-used words of its language model, so predicting an action is just predicting the next word.

## 3 · What the bins cost

Decode the tokens back into deltas and replay them on the robot, open-loop. Compare the end point with the original path, for different numbers of bins.

In [ ]:
reference = at.replay(robot, path[0], deltas)
for n in (256, 16, 4):
    decoded = at.detokenize(at.tokenize(deltas, stats, n), stats, n)
    replayed = at.replay(robot, path[0], decoded)
    print(f"{n:3d} bins: worst single-step error {np.abs(decoded - deltas).max() * 1000:6.3f} mm, "
          f"end point off by {np.linalg.norm(replayed[-1] - reference[-1]) * 1000:6.2f} mm")

The video's 0.78 mm was arithmetic for a 20 cm range: 200 mm ÷ 256. Your range here is much smaller than 20 cm, because each *step* is small, so your bins are much finer. That's the trick: tokenize steps, not positions.

## 4 · The wrong statistics

The third silent failure in the video: decode with a different robot's statistics. Same tokens, different ruler.

In [ ]:
wrong = {"q01": stats["q01"] / 20, "q99": stats["q99"] / 20}    # a robot whose steps are 20x smaller
replayed_wrong = at.replay(robot, path[0], at.detokenize(tokens, wrong))
print(f"intended travel: {np.linalg.norm(path[-1] - path[0]) * 100:.1f} cm")
print(f"with the wrong statistics: {np.linalg.norm(replayed_wrong[-1] - replayed_wrong[0]) * 100:.1f} cm")
cam = {"lookat": [0.62, 0.0, 0.45], "distance": 1.2, "azimuth": 160, "elevation": -45}
robot.set_q(robot.ik(replayed_wrong[-1], q0=robot.ready))
media.show_image(robot.render(560, 360, camera=cam, trail=replayed_wrong, guide=path))

The grey line is where the hand should have gone; the red one is where it went. No error was raised.

## 5 · Try it

1. Clip at the 5th and 95th percentiles instead of the 1st and 99th. Which steps get flattened, and does the end point drift more?
2. Make the trajectory faster (60 steps instead of 120). What happens to the bin width?
3. Decode with statistics that are 3× too *large*. Which way does the robot go wrong now?
4. OpenVLA decodes each bin to its centre rather than its edge. Change `detokenize` to use bin centres and compare.

In [ ]:
# your experiments here